# Diabetes Prediction - Model Training

This notebook builds a machine learning model that predicts whether a patient is likely to have diabetes based on routine medical measurements.

**Dataset:** [Diabetes Dataset](https://www.kaggle.com/datasets/salihacur/diabetes) (Kaggle).

We are running this on Google Colab, so the dataset file downloaded from Kaggle has been uploaded to the Colab session's `/content` folder as `diabetes.csv`.

## 1. Load the dataset

We read `diabetes.csv` into a pandas DataFrame and take a first look at the rows and columns.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

DATA_PATH = "/content/diabetes.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()

Shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## 2. Quick look at the data

We check the data types, the target balance, and summary statistics. In this dataset the `Outcome` column is the target: `1` means the patient has diabetes and `0` means they do not.

In [ ]:
df.info()
print()
print("Target counts (0 = no diabetes, 1 = diabetes):")
print(df["Outcome"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB

Target counts (0 = no diabetes, 1 = diabetes):
Outcome
0    500
1    268
Name: count, dtype: int64


In [ ]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Pregnancies,768.0,3.845052,3.369578,0.000,1.00000,3.0000,6.00000,17.00
Glucose,768.0,120.894531,31.972618,0.000,99.00000,117.0000,140.25000,199.00
BloodPressure,768.0,69.105469,19.355807,0.000,62.00000,72.0000,80.00000,122.00
SkinThickness,768.0,20.536458,15.952218,0.000,0.00000,23.0000,32.00000,99.00
Insulin,768.0,79.799479,115.244002,0.000,0.00000,30.5000,127.25000,846.00
BMI,768.0,31.992578,7.884160,0.000,27.30000,32.0000,36.60000,67.10
DiabetesPedigreeFunction,768.0,0.471876,0.331329,0.078,0.24375,0.3725,0.62625,2.42
Age,768.0,33.240885,11.760232,21.000,24.00000,29.0000,41.00000,81.00
Outcome,768.0,0.348958,0.476951,0.000,0.00000,0.0000,1.00000,1.00


## 3. Split features and target

The `Outcome` column is what we want to predict. The other eight columns are the input features.

We split the data into a training set and a testing set. The model learns from the training set and is evaluated on the testing set, which it has never seen before.

In [ ]:
X = df.drop(columns=["Outcome"])
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows: ", X_test.shape[0])

Training rows: 614
Testing rows:  154


## 4. Build the model

We use a scikit-learn `Pipeline`. A pipeline bundles the preprocessing steps (filling missing values and scaling the numbers) together with the classifier. This matters because the API later has to apply the *exact same* steps before predicting.

In [ ]:
model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(n_estimators=200, random_state=42)),
])

model.fit(X_train, y_train)
print("Model trained.")

Model trained.


## 5. Evaluate on the test set

In [ ]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}\n")
print("Classification report:")
print(classification_report(y_test, y_pred, target_names=["No diabetes", "Diabetes"]))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7468

Classification report:
              precision    recall  f1-score   support

 No diabetes       0.79      0.83      0.81       100
    Diabetes       0.65      0.59      0.62        54

    accuracy                           0.75       154
   macro avg       0.72      0.71      0.72       154
weighted avg       0.74      0.75      0.74       154

Confusion matrix:
[[83 17]
 [22 32]]


## 6. Save the model and its parameters

We save the trained pipeline as `model.joblib`. We also write the model settings and the test accuracy to `metrics.json` so the results are easy to report and to keep for reference.

In [ ]:
model_dir = Path("/content/model")
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "model.joblib"
joblib.dump(model, model_path)

metrics = {
    "dataset": "salihacur/diabetes",
    "model": "RandomForestClassifier",
    "n_estimators": 200,
    "test_size": 0.2,
    "random_state": 42,
    "accuracy": round(float(accuracy), 4),
    "features": list(X.columns),
}
(model_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))

print("Saved model to:", model_path)
print(json.dumps(metrics, indent=2))

Saved model to: /content/model/model.joblib
{
  "dataset": "salihacur/diabetes",
  "model": "RandomForestClassifier",
  "n_estimators": 200,
  "test_size": 0.2,
  "random_state": 42,
  "accuracy": 0.7468,
  "features": [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age"
  ]
}


## 7. Test the saved model

As a final check, we load `model.joblib` back from disk and make a prediction on a single patient. This is the same thing the FastAPI service will do when it receives a request.

In [ ]:
loaded_model = joblib.load(model_path)

sample = X_test.iloc[[0]]
prediction = int(loaded_model.predict(sample)[0])
probability = float(loaded_model.predict_proba(sample)[0][1])

print("Sample patient:")
print(sample.to_dict(orient="records")[0])
print()
print(f"Prediction: {prediction} ({'Diabetes' if prediction == 1 else 'No diabetes'})")
print(f"Probability of diabetes: {probability:.4f}")

Sample patient:
{'Pregnancies': 7, 'Glucose': 159, 'BloodPressure': 64, 'SkinThickness': 0, 'Insulin': 0, 'BMI': 27.4, 'DiabetesPedigreeFunction': 0.294, 'Age': 40}

Prediction: 1 (Diabetes)
Probability of diabetes: 0.6950
